<a href="https://colab.research.google.com/github/dhaneshguptha15253-tech/Fraud-detection-agent/blob/main/integrating_mcp_server_in_skillmap_agent_initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI Model's Instance Creation

In [14]:
!pip install -U langchain-google-genai

In [15]:
!pip install langchain langchain-tavily

In [16]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [17]:
!pip install langchain-mcp-adapters

In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient


In [19]:
client = MultiServerMCPClient(
    {
        "mcp_tavily": {
            "transport": "http",
            "url": "https://backend.composio.dev/v3/mcp/b827882d-eb98-491d-82d1-53f70423271c/mcp?user_id=pg-test-7656991e-a28f-4be4-a413-e7e3a74d48a6",

        }
    }
)

pg-test-7656991e-a28f-4be4-a413-e7e3a74d48a6

TAVILY TOOL CREATION

In [20]:
from langchain_tavily import TavilySearch
from google.colab import userdata
tavily_api_key = userdata.get('TAVILY_API_KEY')
skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)

Rapid Api website's JSearch for getting Live Job Links

In [21]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill. Falls back to mock listings if API Key is unavailable."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    try:
        rapidapi_key = userdata.get('RAPIDAPI_KEY')
    except Exception:
        print("RAPIDAPI_KEY not found in Colab Secrets. Falling back to simulated live job listings...")
        rapidapi_key = None

    if not rapidapi_key:
        # High quality fallback simulated live job openings for India
        return [
            {
                "title": f"Generative AI Engineer",
                "company": "TechInnovate Solutions",
                "location": "Bengaluru, India",
                "apply_link": "https://www.linkedin.com/jobs"
            },
            {
                "title": "Machine Learning Engineer (LLMs)",
                "company": "AI Frontier Labs",
                "location": "Hyderabad, India",
                "apply_link": "https://www.naukri.com"
            },
            {
                "title": "Junior GenAI Developer",
                "company": "Enterprise Automation Corp",
                "location": "Pune, India",
                "apply_link": "https://www.indeed.com"
            }
        ]

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }
    try:
        response = requests.get(url, headers=headers, params=querystring)
        data = response.json()
        jobs = data.get("data", [])
        print(f"Found {len(jobs)} jobs\n")

        result = []
        for job in jobs:
            result.append({
                "title": job.get("job_title"),
                "company": job.get("employer_name"),
                "location": job.get("job_city"),
                "apply_link": job.get("job_apply_link")
            })
        return result
    except Exception as e:
        print(f"Error fetching from API: {e}. Returning simulated listings.")
        return [
            {
                "title": f"Generative AI Specialist",
                "company": "Global Solutions India",
                "location": "Noida, India",
                "apply_link": "https://www.linkedin.com/jobs"
            }
        ]

In [22]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- search tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""


In [23]:
from langchain.agents import create_agent

def skill_map_agent():
    # Use your local tools: Tavily Search (skill_demand_tool) and JSearch (search_jobs)
    all_tools = [skill_demand_tool, search_jobs]

    print("Creating agent with local tools...")
    agent = create_agent(
        model=model,
        tools=all_tools,
        debug=True,
        system_prompt=system_prompt
    )

    # User query
    user_query = "What's the demand for generative AI in the industry and show me related job openings in India"

    # Invoke agent synchronously
    print("Invoking agent...")
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": user_query}
        ]
    })

    # Print response
    print("\n--- Agent Response ---")
    print(response["messages"][-1].content)

# Run the agent
skill_map_agent()

Creating agent with local tools...
Invoking agent...
[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='25875b5a-f203-49ca-b686-050a8b92f0df')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "generative AI industry demand"}'}, '__gemini_function_call_thought_signatures__': {'e4621f82-f38b-4f80-9544-5e1cc0d65dbe': 'CukDAWkUfRM22ZSvW0JG+7D7zPgbXyO43WV2+3t0Sr2ck+ubAeD0jmy7UjiPBVb0rKT+kYHDWVPEErs666eh3vSIEjvz0G0P4kNb8NSNZ81pPE4VqXFk4EIPil+HTn/wh44ADvE4OeqqEqtP1DJmJPWL0nMOJju7pOi9LiPrBfv7FP5X86emR9GIo21OJgjMgUeGOa14jggLm1CUFdQ/nLiAWLO7K/cKgm3njUO0aDvMZikWniHhv/UBZYPEvLHNRvH1FH0qspXy6fnTHQD+ff1SO8PophXw6+cEJ71iZhpXfTbQvevH05csBToVthTERvqUR0HLkCVW+f4RLe0UbAJSNfUyhVbWlQMfVg5nuxlsNWg8JgPIJloO/8Q1VeZ5R3/FTKe8Cmlu34hajt9qX7OH10PoYUnDOF5UEvhoniipyGQBCZ+3fCR7j6FsYo1RMf

from langchain.agents import create_agent

def skill_map_agent():

  agent = create_agent(
    model=model,
  tools=[skill_demand_tool, search_jobs],
  system_prompt=system_prompt,
  debug=True

  )
  user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

  response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
  })
  print(response["messages"][-1].content[0]["text"])


skill_map_agent()